In [1]:
import torch
import torch.nn.functional as F   
from diffusers import StableDiffusionPipeline, UNet2DConditionModel

device = 'cuda:3'



In [2]:

basemodel_id = "CompVis/stable-diffusion-v1-4"
torch_dtype = torch.bfloat16

pipe = StableDiffusionPipeline.from_pretrained(basemodel_id, torch_dtype=torch_dtype).to(device)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/home/nessessence/anaconda3/envs/uul/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:

def kv_angular_loss(
    unet_u,         # unlearned UNet (trainable)
    unet_0,         # frozen reference UNet
    p_e,            # erased prompt embedding [B, T, 768]
    p_g,            # generic prompt embedding [B, T, 768]
    m_excl: float,
    m_incl: float,
    layer_filter="attn2",  # cross-attention in diffusers SD1.4
    use_v: bool = True,
):
    """
    Angular exclusion + inclusion loss in KV projection space (unsquared hinge).

      L_excl = max(cos(W_u p_e, W_0 p_e) - m_excl, 0)
      L_incl = max(m_incl - cos(W_u p_e, W_0 p_g), 0)

    Gradients flow ONLY into unet_u.
    Returns: (L_excl, L_incl, L_ang)
    """

    params_u = dict(unet_u.named_parameters())
    params_0 = dict(unet_0.named_parameters())

    excl_terms = []
    incl_terms = []
    matched_layers = 0

    for name, W_u in params_u.items():
        if layer_filter not in name:
            continue
        if not (name.endswith("to_k.weight") or (use_v and name.endswith("to_v.weight"))):
            continue
        if name not in params_0:
            continue

        # Corresponding frozen weight
        W_0 = params_0[name].detach()

        # Bias (optional)
        b_name = name.replace(".weight", ".bias")
        b_u = params_u.get(b_name, None)
        b_0 = params_0.get(b_name, None)
        if b_0 is not None:
            b_0 = b_0.detach()

        # Linear projections
        W_u_e = F.linear(p_e, W_u, b_u)

        with torch.no_grad():
            W_0_e = F.linear(p_e, W_0, b_0)
            W_0_g = F.linear(p_g, W_0, b_0)

        # Mean-pool tokens: [B, T, D] → [B, D]
        W_u_e = W_u_e.mean(dim=1)
        W_0_e = W_0_e.mean(dim=1)
        W_0_g = W_0_g.mean(dim=1)

        # Cosine similarities
        cos_excl = F.cosine_similarity(W_u_e, W_0_e, dim=-1)  # [B]
        cos_incl = F.cosine_similarity(W_u_e, W_0_g, dim=-1)  # [B]

        # Unsquared hinge losses (batch-mean)
        excl_terms.append(torch.clamp(cos_excl - m_excl, min=0.0).mean())
        incl_terms.append(torch.clamp(m_incl - cos_incl, min=0.0).mean())
        matched_layers += 1

    if matched_layers == 0:
        zero = torch.tensor(0.0, device=p_e.device)
        return zero, zero, zero

    L_excl = torch.stack(excl_terms).mean()
    L_incl = torch.stack(incl_terms).mean()
    L_ang  = L_excl + L_incl

    return L_excl, L_incl, L_ang


In [4]:
prompt = ['a photo of obama','a photo of person']
text_embedding, _ = pipe.encode_prompt(prompt=prompt,
                                        device=device,
                                        num_images_per_prompt=1,
                                        do_classifier_free_guidance=False)  
p_e, p_g = text_embedding.chunk(2, dim=0)

In [5]:
kv_angular_loss(
    unet_u=pipe.unet,
    unet_0=pipe.unet,
    p_e=p_e,
    p_g=p_g,
    m_excl=0.2,
    m_incl=0.5,
    layer_filter="attn2",
)

(tensor(0.6406, device='cuda:3', dtype=torch.bfloat16, grad_fn=<MeanBackward0>),
 tensor(0., device='cuda:3', dtype=torch.bfloat16, grad_fn=<MeanBackward0>),
 tensor(0.6406, device='cuda:3', dtype=torch.bfloat16, grad_fn=<AddBackward0>))